In [95]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [96]:
import torch 
import matplotlib.pyplot as plt

device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [97]:
from agent import AgentMLP , AgentCNN
from environment import  snake_environment

In [98]:
snake_env1=snake_environment(5,5)

In [99]:
state=snake_env1.get_state()
state.shape

torch.Size([5, 5, 4])

In [100]:
snake_env1.render()
snake_env1.step(0)

_ _ _ _ _
_ _ _ _ _
_ O _ _ _
_ X _ _ _
_ _ _ _ F


(tensor([[[1., 0., 0., 0.],
          [1., 0., 0., 0.],
          [1., 0., 0., 0.],
          [1., 0., 0., 0.],
          [1., 0., 0., 0.]],
 
         [[1., 0., 0., 0.],
          [1., 0., 0., 0.],
          [1., 0., 0., 0.],
          [0., 0., 0., 1.],
          [0., 1., 0., 0.]],
 
         [[1., 0., 0., 0.],
          [1., 0., 0., 0.],
          [1., 0., 0., 0.],
          [1., 0., 0., 0.],
          [1., 0., 0., 0.]],
 
         [[1., 0., 0., 0.],
          [1., 0., 0., 0.],
          [1., 0., 0., 0.],
          [1., 0., 0., 0.],
          [1., 0., 0., 0.]],
 
         [[1., 0., 0., 0.],
          [1., 0., 0., 0.],
          [1., 0., 0., 0.],
          [1., 0., 0., 0.],
          [0., 0., 1., 0.]]]),
 -0.1,
 False)

In [101]:
hidden_size=2048
snake_agent1=AgentMLP(snake_env1.d_model*snake_env1.width*snake_env1.height,hidden_size)



#snake_agent1 training
score_log=[]
score_mean_log=[]
batch=500
render_every=1000
num_games=10000
batch_size=500
update_target_every=500


for epoch in range(num_games+1):
    score=snake_env1.snake.score
    snake_env1.reset()
    if (epoch + 1) % update_target_every == 0:
        snake_agent1.update_target_model()

        
    if   epoch>100 and epoch%render_every==0:
        print(f"GAME NUMBER {epoch}")
        
        original_epsilon = snake_agent1.epsilon
        snake_agent1.epsilon = 0
        
        while not snake_env1.gameover : 
            state=snake_env1.get_state()
            state=state.flatten()
            action=snake_agent1.get_action(state)
            next_state , reward , done =snake_env1.step(action)
            next_state=next_state.flatten()
            snake_env1.render()
            print("===================")
        snake_agent1.epsilon = original_epsilon
    else:
        while not snake_env1.gameover :
            state=snake_env1.get_state()
            state=state.flatten()
            action=snake_agent1.get_action(state)
            next_state , reward , done =snake_env1.step(action)
            next_state=next_state.flatten()
            snake_agent1.remember(state,action,reward,next_state,done)
            snake_agent1.replay(batch_size)
    
    score_log.append(score)
    print(f"score for game {epoch}: {score}")
    if (epoch + 1)%batch==0:
        mean=sum(score_log)/len(score_log)
        score_mean_log.append(mean)
        score_log=[]

plt.figure(figsize=(12, 6))
plt.plot(score_mean_log, label='score mean')
plt.xlabel(f'Batch of {batch} Games')
plt.ylabel(f'average score')
plt.grid(True)
plt.show()

    


In [102]:
state_dict = torch.load("models/qNetwork.pth", weights_only=True,map_location=torch.device(device))
snake_agent1.model.load_state_dict(state_dict)

<All keys matched successfully>

In [103]:

#snake_agent1 training

num_games=1
snake_agent1.epsilon = 0

for epoch in range(num_games):
    score=snake_env1.snake.score
    snake_env1.reset()
    print(f"GAME NUMBER {epoch}")

    while not snake_env1.gameover : 
        state=snake_env1.get_state()
        state=state.flatten()
        
        action=snake_agent1.get_action(state)
        next_state , reward , done =snake_env1.step(action)
        snake_env1.render()
        print("===================")
    
    print(f"score for game {epoch}: {score}")

    


GAME NUMBER 0
_ _ _ _ _
X O _ _ _
_ _ _ _ _
_ _ _ _ _
F _ _ _ _
_ _ _ _ _
O _ _ _ _
X _ _ _ _
_ _ _ _ _
F _ _ _ _
_ _ _ _ _
_ _ _ _ _
O _ _ _ _
X _ _ _ _
F _ _ _ _
_ _ _ _ _
_ _ _ _ _
O _ _ _ _
O _ _ _ _
X _ _ _ F
_ _ _ _ _
_ _ _ _ _
_ _ _ _ _
O _ _ _ _
O X _ _ F
_ _ _ _ _
_ _ _ _ _
_ _ _ _ _
_ _ _ _ _
O O X _ F
_ _ _ _ _
_ _ _ _ _
_ _ _ _ _
_ _ _ _ _
_ O O X F
_ _ _ _ _
_ _ F _ _
_ _ _ _ _
_ _ _ _ _
_ O O O X
_ _ _ _ _
_ _ F _ _
_ _ _ _ _
_ _ _ _ X
_ _ O O O
_ _ _ _ _
_ _ F _ _
_ _ _ _ X
_ _ _ _ O
_ _ _ O O
_ _ _ _ _
_ _ F _ X
_ _ _ _ O
_ _ _ _ O
_ _ _ _ O
_ _ _ _ _
_ _ F X O
_ _ _ _ O
_ _ _ _ O
_ _ _ _ _
_ _ _ _ _
_ _ X O O
_ _ _ _ O
_ _ _ _ O
_ _ F _ _
_ _ _ _ _
_ X O O O
_ _ _ _ O
_ _ _ _ _
_ _ F _ _
_ _ _ _ _
_ O O O O
_ X _ _ _
_ _ _ _ _
_ _ F _ _
_ _ _ _ _
_ O O O _
_ O _ _ _
_ X _ _ _
_ _ F _ _
_ _ _ _ _
_ O O _ _
_ O _ _ _
_ O _ _ _
_ X F _ _
_ _ F _ _
_ O O _ _
_ O _ _ _
_ O _ _ _
_ O X _ _
_ _ F _ _
_ O _ _ _
_ O _ _ _
_ O _ _ _
_ O O X _
_ _ F _ _
_ _ _ _ _
_ O _ _ _
_ O _ 

In [104]:
snake_agent2=AgentCNN(snake_env1.d_model,snake_env1.width,snake_env1.height,hidden_size=556)

In [ ]:
import matplotlib.pyplot as plt

# Existing Hyperparameters
score_log = []
score_mean_log = []
logging_batch = 500
render_every = 1000
num_games = 10000
batch_size = 256
update_target_every = 512

# Tracking Variables
best_score = -1
loss_log = []
loss_mean_log = []
death_body_count = 0
death_wall_count = 0
death_ratio_log = []

# New Tracking Variables for Game Length
length_log = []
length_mean_log = []

for epoch in range(num_games + 1):
    score = snake_env1.snake.score

    # 1. Track and print Best Score
    if score > best_score and epoch > 0:
        best_score = score
        print(f">>> NEW BEST SCORE: {best_score} at Game {epoch - 1} <<<")

    snake_env1.reset()
    game_steps = 0

    # update Target Model
    if (epoch + 1) % update_target_every == 0:
        snake_agent2.update_target_model()

    # render the game    
    if epoch > 100 and epoch % render_every == 0:
        print(f"GAME NUMBER {epoch}")
        original_epsilon = snake_agent2.epsilon
        snake_agent2.epsilon = 0
        
        while not snake_env1.gameover: 
            state = snake_env1.get_state()
            action = snake_agent2.get_action(state.unsqueeze(0))
            next_state, reward, done = snake_env1.step(action)
            snake_env1.render()
            game_steps += 1
            print("===================")
        snake_agent2.epsilon = original_epsilon
        
    # train normally
    else:
        game_loss = 0
        while not snake_env1.gameover:
            state = snake_env1.get_state()
            action = snake_agent2.get_action(state.unsqueeze(0))
            next_state, reward, done = snake_env1.step(action)
            snake_agent2.remember(state, action, reward, next_state, done)
            game_steps += 1

            # 2. Track Loss
            if game_steps%4==0:
                loss = snake_agent2.replay(batch_size)
                if loss is not None:
                    game_loss += loss
        
        # Calculate average loss for this specific game
        if game_steps > 0:
            loss_log.append(game_loss / game_steps)

    # 3. Track Death Reason
    if hasattr(snake_env1, 'death_reason'):
        if snake_env1.death_reason == 'wall':
            death_wall_count += 1
        elif snake_env1.death_reason == 'body':
            death_body_count += 1

    score_log.append(score)
    length_log.append(game_steps)
    print(f"score for game {epoch}: {score} | steps: {game_steps}")

    # Logging batch updates
    if (epoch + 1) % logging_batch == 0:
        # Score Mean
        mean = sum(score_log) / len(score_log)
        score_mean_log.append(mean)
        score_log = []

        # Loss Mean
        if loss_log:
            loss_mean = sum(loss_log) / len(loss_log)
            loss_mean_log.append(loss_mean)
            loss_log = []
        else:
            loss_mean_log.append(0)
            
        # Game Length Mean
        if length_log:
            length_mean = sum(length_log) / len(length_log)
            length_mean_log.append(length_mean)
            length_log = []
        else:
            length_mean_log.append(0)
        
        print(f"*** Batch {epoch + 1} | Avg Loss: {loss_mean_log[-1]:.4f} | Avg Length: {length_mean_log[-1]:.1f} steps ***")

        # Death Ratio
        total_deaths = death_body_count + death_wall_count
        if total_deaths > 0:
            death_ratio_log.append((death_body_count / total_deaths, death_wall_count / total_deaths))
        else:
            death_ratio_log.append((0, 0))
        
        # Reset death counts for the next batch
        death_body_count = 0
        death_wall_count = 0


# 4. Plotting all metrics as subplots
fig, axs = plt.subplots(4, 1, figsize=(12, 24))

# Plot 1: Score Mean
axs[0].plot(score_mean_log, label='Score Mean', color='blue', marker='o')
axs[0].set_title('Average Score per Batch')
axs[0].set_xlabel(f'Batch of {logging_batch} Games')
axs[0].set_ylabel('Average Score')
axs[0].grid(True)
axs[0].legend()

# Plot 2: Average Loss
axs[1].plot(loss_mean_log, label='Loss Mean', color='red', marker='o')
axs[1].set_title('Average Loss per Batch')
axs[1].set_xlabel(f'Batch of {logging_batch} Games')
axs[1].set_ylabel('Average Loss')
axs[1].grid(True)
axs[1].legend()

# Plot 3: Death Reason Ratio (Stacked Bar Chart)
body_ratios = [r[0] for r in death_ratio_log]
wall_ratios = [r[1] for r in death_ratio_log]
batches = range(len(death_ratio_log))

axs[2].bar(batches, body_ratios, label='Ran into Body', color='orange')
axs[2].bar(batches, wall_ratios, bottom=body_ratios, label='Ran into Wall', color='gray')
axs[2].set_title('Death Reason Ratio per Batch')
axs[2].set_xlabel(f'Batch of {logging_batch} Games')
axs[2].set_ylabel('Ratio')
axs[2].legend()

# Plot 4: Average Game Length
axs[3].plot(length_mean_log, label='Length Mean (Steps)', color='green', marker='o')
axs[3].set_title('Average Game Length per Batch')
axs[3].set_xlabel(f'Batch of {logging_batch} Games')
axs[3].set_ylabel('Average Steps per Game')
axs[3].grid(True)
axs[3].legend()

plt.tight_layout()
plt.show()

score for game 0: 13 | steps: 12
>>> NEW BEST SCORE: 1 at Game 0 <<<
score for game 1: 1 | steps: 1
score for game 2: 0 | steps: 4
score for game 3: 0 | steps: 2
score for game 4: 0 | steps: 3
score for game 5: 0 | steps: 6
score for game 6: 1 | steps: 1
score for game 7: 0 | steps: 5
score for game 8: 0 | steps: 3
score for game 9: 0 | steps: 5
score for game 10: 0 | steps: 11
score for game 11: 0 | steps: 8
score for game 12: 0 | steps: 1
score for game 13: 0 | steps: 2
score for game 14: 0 | steps: 2
score for game 15: 0 | steps: 2
score for game 16: 0 | steps: 14
score for game 17: 0 | steps: 6
score for game 18: 1 | steps: 1
score for game 19: 0 | steps: 5
score for game 20: 0 | steps: 10
score for game 21: 0 | steps: 4
score for game 22: 0 | steps: 4
score for game 23: 0 | steps: 12
score for game 24: 0 | steps: 4
score for game 25: 0 | steps: 2
score for game 26: 0 | steps: 2
score for game 27: 0 | steps: 12
score for game 28: 0 | steps: 4
score for game 29: 0 | steps: 7
score f

KeyboardInterrupt: 

In [ ]:
len(snake_agent2.memory)

11